In [ ]:
# === Session start: mount Drive + pull latest code from GitHub ===
from google.colab import drive
drive.mount('/content/drive')

%cd /content/drive/MyDrive/LoR-LUT

# Drive 这份 clone 只接收 pull,不应该有 local commit
!git status -sb

# --ff-only: Drive 那份意外有 local commit 的话立即 abort,而不是产生 merge
!git pull --ff-only origin main

# 确认拉到最新 commit
!git log --oneline -3

# LoR-IA-3DLUT • Colab Quickstart

按顺序执行每个单元即可开始训练。

In [ ]:
!nvidia-smi
import os

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
%cd /content/drive/MyDrive/LoR-LUT

In [ ]:
!pip install -r requirements.txt

## 准备数据
将你的数据整理为 `/dataset_root/{train,val}/{input,gt}` 结构，然后设置到下面变量。

In [ ]:
DATA_ROOT = '/content/drive/MyDrive/datasets/Adobe5kJPG_c'  # 修改成你的路径
WORK_DIR = '/content/drive/MyDrive/LoR-LUT/runs/fivek_lor'

In [ ]:
import yaml
with open("config/default.yaml") as f:
    cfg = yaml.safe_load(f)
print(cfg["loss"])
print({k:type(v).__name__ for k,v in cfg["loss"].items()})

In [ ]:
#check dataset preparation
from data.paired_folder import PairedFolderDataset
ds = PairedFolderDataset(root=DATA_ROOT, split="train", in_dir="input", gt_dir="gt", patch=0, augment=False)
for i in range(5):
    s = ds[i]
    print(s["name"], (s["img_in"]-s["img_gt"]).abs().mean().item())

In [ ]:
import torch
import yaml
from core.core_lut import LoRIA3DLUT
from data.paired_folder import PairedFolderDataset
from losses.delta_e import delta_e_2000_srgb

# 1. 加载配置和模型
ckpt_path = "/content/drive/MyDrive/LoR-LUT/runs/fivek_lor/best.ckpt"  # 或者 last.ckpt，或你训练到16000步左右的ckpt
device = "cuda" if torch.cuda.is_available() else "cpu"

print(f"Loading checkpoint: {ckpt_path}")
ckpt = torch.load(ckpt_path, map_location=device)
cfg = ckpt["cfg"]

G = cfg["model"]["G"]
K = cfg["model"]["K"]
R = cfg["model"]["R"]

model = LoRIA3DLUT(G=G, K=K, R=R).to(device)
model.load_state_dict(ckpt["state_dict"], strict=True)
model.eval()

print(f"Model loaded: G={G}, K={K}, R={R}")

# 2. 加载数据集
root = "/content/drive/MyDrive/datasets/Adobe5kJPG_c"  # 改成你的数据集路径
ds = PairedFolderDataset(
    root=root,
    split="train",
    in_dir="input",
    gt_dir="gt",
    exts=tuple(cfg["data"]["ext"]),
    patch=512,
    augment=False
)

print(f"Dataset loaded: {len(ds)} samples")

# 3. 测试几个样本
num_test = 5
for i in range(min(num_test, len(ds))):
    sample = ds[i]
    img_lr = sample["img_lr"].unsqueeze(0).to(device)  # [1,3,256,256]
    img_in = sample["img_in"].unsqueeze(0).to(device)  # [1,3,512,512]
    img_gt = sample["img_gt"].unsqueeze(0).to(device)

    with torch.no_grad():
        pred, _ = model(img_lr, img_in)

        pred_min = pred.min().item()
        pred_max = pred.max().item()
        pred_mean = pred.mean().item()

        print(f"\n[Sample {i}] {sample['name']}")
        print(f"  pred range: [{pred_min:.4f}, {pred_max:.4f}], mean={pred_mean:.4f}")

        # 检查是否超出 [0,1]
        out_of_range = (pred < 0).any() or (pred > 1).any()
        if out_of_range:
            print(f"  ⚠️  pred 超出 [0,1] 范围!")

        # 测试 ΔE2000（不clamp）
        try:
            de_no_clamp = delta_e_2000_srgb(pred, img_gt)
            de_mean = de_no_clamp.mean().item()
            has_nan = torch.isnan(de_no_clamp).any().item()
            has_inf = torch.isinf(de_no_clamp).any().item()
            print(f"  ΔE2000 (no clamp): mean={de_mean:.4f}, has_nan={has_nan}, has_inf={has_inf}")
        except Exception as e:
            print(f"  ❌ ΔE2000 (no clamp) 崩溃: {e}")

        # 测试 ΔE2000（clamp到[0,1]）
        try:
            de_clamped = delta_e_2000_srgb(pred.clamp(0,1), img_gt.clamp(0,1))
            de_mean_c = de_clamped.mean().item()
            has_nan_c = torch.isnan(de_clamped).any().item()
            print(f"  ΔE2000 (clamped):  mean={de_mean_c:.4f}, has_nan={has_nan_c}")
        except Exception as e:
            print(f"  ❌ ΔE2000 (clamped) 崩溃: {e}")

print("\n✅ 验证完成")

In [ ]:
# 在 Colab 里运行
import torch
from data.paired_folder import PairedFolderDataset
from core.core_lut import LoRIA3DLUT
from utils.metrics import psnr

root = "/content/drive/MyDrive/datasets/Adobe5kJPG_c"
device = "cuda" if torch.cuda.is_available() else "cpu"

# 1. 检查验证集大小
ds_val = PairedFolderDataset(root, "val", "input", "gt",
                              exts=(".jpg",".jpeg",".png",".tif",".tiff"),
                              patch=0, augment=False)
print(f"✅ 验证集图片数: {len(ds_val)}")

# 2. 检查前几张图的 input vs gt 差异
print("\n验证集前5张的 input-gt 差异:")
for i in range(min(5, len(ds_val))):
    s = ds_val[i]
    mae = (s["img_in"] - s["img_gt"]).abs().mean().item()
    print(f"  {s['name']}: MAE={mae:.4f}")

# 3. 加载模型，看"恒等输入"的 PSNR 是多少
ckpt = torch.load("runs/fivek_lor/best.ckpt", map_location=device)
cfg = ckpt["cfg"]
model = LoRIA3DLUT(G=cfg["model"]["G"], K=cfg["model"]["K"], R=cfg["model"]["R"]).to(device)
model.load_state_dict(ckpt["state_dict"])
model.eval()

with torch.no_grad():
    s = ds_val[0]
    img_lr = s["img_lr"].unsqueeze(0).to(device)
    img_in = s["img_in"].unsqueeze(0).to(device)
    img_gt = s["img_gt"].unsqueeze(0).to(device)

    # 模型预测
    pred, _ = model(img_lr, img_in)
    psnr_pred = psnr(pred.clamp(0,1), img_gt.clamp(0,1)).item()

    # 恒等输出（直接用 input）
    psnr_identity = psnr(img_in.clamp(0,1), img_gt.clamp(0,1)).item()

    print(f"\n第一张图的 PSNR:")
    print(f"  模型输出: {psnr_pred:.2f}")
    print(f"  恒等输出: {psnr_identity:.2f}")
    print(f"  差异: {psnr_pred - psnr_identity:.2f}")

In [ ]:
!python train.py \
  --data.root "$DATA_ROOT" \
  --work_dir "$WORK_DIR" \
  --cfg config/default.yaml

## 验证与可视化

In [ ]:

!python evaluate.py \
  --data.root "$DATA_ROOT/val" \
  --ckpt "$WORK_DIR/best.ckpt" \
  --out_dir "$WORK_DIR/val_vis"

In [ ]:
import torch
ckpt = torch.load("/content/drive/MyDrive/LoR-LUT/runs/fivek_lor/best.ckpt", map_location="cpu")
print(ckpt.keys())  # dict_keys(['state_dict','optimizer','cfg','iter','best'])
print("iter:", ckpt["iter"], "best:", ckpt["best"])
# 看权重名和形状（前10个）
for i,(k,v) in enumerate(ckpt["state_dict"].items()):
    print(k, tuple(v.shape))
    # if i>=9: break